# Bloom Embedding × DCN — Criteo CTR

**Embedding:** Serrà & Karatzoglou — Bloom Embeddings for Sparse Binary I/O (RecSys 2017).

**Backbone:** DCN (kiến trúc theo `criteo-benchmark/model/dcn.py`).

**Quy ước:** categorical embedding 64 chiều. DCN/DeepFM concat trực tiếp 13 cột integer (raw, không embedding).

## 1. Imports & Config

In [ ]:
import os, math, time, gc, random
from dataclasses import dataclass, field
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, log_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('Torch:', torch.__version__, '| Device:', device)

In [ ]:
@dataclass
class Cfg:
    sample_rows: int = 2_000_000
    val_ratio: float = 0.1
    embed_dim: int = 64
    batch_size: int = 4096
    epochs: int = 3
    lr: float = 1e-3
    weight_decay: float = 1e-6
    bloom_m: int = 100_000
    bloom_k: int = 4
    bloom_threshold: int = 10_000
    cross_layers: int = 3
    hidden_dims: List[int] = field(default_factory=lambda: [400, 400, 256])

cfg = Cfg()
print(cfg)

## 2. Load dữ liệu Criteo cleaned

In [ ]:
def find_files(roots):
    for root in roots:
        if os.path.isdir(root):
            paths = []
            for r, _, fs in os.walk(root):
                for f in fs:
                    if f.lower().endswith(('.parquet', '.csv', '.tsv', '.txt')):
                        paths.append(os.path.join(r, f))
            if paths:
                return sorted(paths)
    return []

def read_any(p, nrows=None):
    if p.endswith('.parquet'):
        df = pd.read_parquet(p)
        return df.head(nrows) if nrows else df
    sep = '\t' if p.endswith(('.tsv', '.txt')) else ','
    return pd.read_csv(p, sep=sep, nrows=nrows)

files = find_files([
    '/kaggle/input/datasets/huy291/criteo-cleaned-data',
    '/kaggle/input/criteo-cleaned-data',
    '/kaggle/input',
])
assert files, 'Không tìm thấy dataset Criteo cleaned'
train_path = next((f for f in files if 'train' in os.path.basename(f).lower()), max(files, key=os.path.getsize))
print('Using:', train_path)

t0 = time.time()
df = read_any(train_path, nrows=cfg.sample_rows if cfg.sample_rows > 0 else None)
print(f'Loaded {len(df):,} rows in {time.time()-t0:.1f}s')

## 3. Tiền xử lý

In [ ]:
# Phát hiện cột
cols = list(df.columns); lower = {c: c.lower() for c in cols}
label_col = next((c for c in cols if lower[c] in ('label', 'click', 'target', 'y')), cols[0])
num_cols = sorted([c for c in cols if c != label_col and lower[c].startswith(('i', 'int', 'num')) and any(ch.isdigit() for ch in c)],
                  key=lambda x: int(''.join(ch for ch in x if ch.isdigit()) or 0))
cat_cols = sorted([c for c in cols if c != label_col and lower[c].startswith(('c', 'cat')) and any(ch.isdigit() for ch in c)],
                  key=lambda x: int(''.join(ch for ch in x if ch.isdigit()) or 0))
print(f'label={label_col} | num={len(num_cols)} | cat={len(cat_cols)}')

# Numerical: log1p + standardize
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0)
    df[c] = np.log1p(df[c].clip(lower=0))
if num_cols:
    mu = df[num_cols].mean(); sd = df[num_cols].std().replace(0, 1)
    df[num_cols] = (df[num_cols] - mu) / sd

# Categorical: factorize → int64 không âm
vocab_sizes = {}
for c in cat_cols:
    if not pd.api.types.is_integer_dtype(df[c]):
        codes, _ = pd.factorize(df[c], sort=False)
        df[c] = codes
    df[c] = df[c].fillna(-1).astype(np.int64) + 1
    vocab_sizes[c] = int(df[c].max()) + 1
print(f'Tổng vocab: {sum(vocab_sizes.values()):,}')

In [ ]:
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
n_val = int(len(df) * cfg.val_ratio)
val_df = df.iloc[:n_val].copy(); train_df = df.iloc[n_val:].copy()
del df; gc.collect()

def to_tensors(d):
    xn = torch.from_numpy(d[num_cols].to_numpy(dtype=np.float32)) if num_cols else torch.zeros(len(d), 0)
    xc = torch.from_numpy(d[cat_cols].to_numpy(dtype=np.int64))
    y = torch.from_numpy(d[label_col].to_numpy(dtype=np.float32))
    return xn, xc, y

xn_tr, xc_tr, y_tr = to_tensors(train_df)
xn_va, xc_va, y_va = to_tensors(val_df)
del train_df, val_df; gc.collect()

train_loader = DataLoader(TensorDataset(xn_tr, xc_tr, y_tr), batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(TensorDataset(xn_va, xc_va, y_va), batch_size=cfg.batch_size*2, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(xn_tr):,} | Val: {len(xn_va):,}')

## 4. Định nghĩa Bloom Embedding

In [ ]:
class BloomEmbedding(nn.Module):
    def __init__(self, vocab_size, m, k, embed_dim, agg='sum', seed=0):
        super().__init__()
        self.m = min(m, vocab_size); self.k = k; self.agg = agg
        self.weight = nn.Parameter(torch.empty(self.m, embed_dim))
        nn.init.normal_(self.weight, std=0.01)
        rng = np.random.default_rng(seed)
        H = rng.integers(0, self.m, size=(vocab_size, k), dtype=np.int64)
        self.register_buffer('H', torch.from_numpy(H), persistent=False)

    def forward(self, idx):
        positions = self.H[idx]
        embeds = F.embedding(positions, self.weight)
        return embeds.sum(dim=-2) if self.agg == 'sum' else embeds.mean(dim=-2)


def build_cat_embeddings(vocab_sizes_dict, cfg):
    mods = []
    for i, (c, v) in enumerate(vocab_sizes_dict.items()):
        if v <= cfg.bloom_threshold:
            e = nn.Embedding(v, cfg.embed_dim); nn.init.normal_(e.weight, std=0.01)
            mods.append(e)
        else:
            mods.append(BloomEmbedding(v, cfg.bloom_m, cfg.bloom_k, cfg.embed_dim, seed=i))
    return nn.ModuleList(mods)

## 5. Mô hình DCN với embedding thay thế

In [ ]:
class CrossNetwork(nn.Module):
    def __init__(self, input_dim, num_layers):
        super().__init__()
        self.cross_weights = nn.ParameterList([nn.Parameter(torch.randn(input_dim)) for _ in range(num_layers)])
        self.cross_bias = nn.ParameterList([nn.Parameter(torch.zeros(input_dim)) for _ in range(num_layers)])

    def forward(self, x0):
        x_l = x0
        for w, b in zip(self.cross_weights, self.cross_bias):
            xl_w = torch.matmul(x_l, w).unsqueeze(1)
            x_l = x0 * xl_w + b + x_l
        return x_l


class DCN(nn.Module):
    def __init__(self, num_dense_features, vocab_sizes, embed_dim, cross_layers, hidden_dims, cfg):
        super().__init__()
        self.embeddings = build_cat_embeddings(vocab_sizes, cfg)
        input_dim = num_dense_features + len(vocab_sizes) * embed_dim
        self.cross_net = CrossNetwork(input_dim, cross_layers)
        layers, in_dim = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(0.3)]
            in_dim = h
        self.deep_net = nn.Sequential(*layers)
        self.fc = nn.Linear(input_dim + in_dim, 1)

    def forward(self, dense_x, sparse_x):
        emb_x = torch.cat([emb(sparse_x[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)
        x0 = torch.cat([dense_x, emb_x], dim=1)
        cross_out = self.cross_net(x0)
        deep_out = self.deep_net(x0)
        combined = torch.cat([cross_out, deep_out], dim=1)
        return torch.sigmoid(self.fc(combined)).squeeze(-1)

## 6. Hàm train & evaluate

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, ys = [], []
    for xn, xc, y in loader:
        xn, xc = xn.to(device, non_blocking=True), xc.to(device, non_blocking=True)
        out = model(xn, xc)
        preds.append(out.detach().cpu().numpy())  # đã sigmoid
        ys.append(y.numpy())
    p = np.concatenate(preds); t = np.concatenate(ys)
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return {'auc': roc_auc_score(t, p), 'logloss': log_loss(t, p)}


def train_model(model, train_loader, val_loader, cfg):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    crit = nn.BCELoss()
    history = []
    for ep in range(1, cfg.epochs + 1):
        model.train(); t0 = time.time(); running = 0.0; nb = 0
        for xn, xc, y in train_loader:
            xn = xn.to(device, non_blocking=True)
            xc = xc.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            out = model(xn, xc)
            loss = crit(out, y)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            running += loss.item(); nb += 1
        elapsed = time.time() - t0
        m = evaluate(model, val_loader)
        history.append({'epoch': ep, 'train_loss': running/nb, 'time_s': elapsed, **m})
        print(f"Ep {ep:02d} | train_loss={running/nb:.4f} | AUC={m['auc']:.4f} | LogLoss={m['logloss']:.4f} | {elapsed:.1f}s")
    return history

## 7. Huấn luyện Bloom Embedding × DCN

In [ ]:
torch.cuda.empty_cache(); gc.collect()
model = DCN(len(num_cols), vocab_sizes, cfg.embed_dim, cfg.cross_layers, cfg.hidden_dims, cfg)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Bloom × DCN | tổng params: {n_params:,}')
history = train_model(model, train_loader, val_loader, cfg)

## 8. Tóm tắt kết quả

In [ ]:
best = max(history, key=lambda h: h['auc'])
summary = pd.DataFrame([{
    'Backbone': 'DCN',
    'Embedding': 'Bloom',
    'Best AUC': round(best['auc'], 4),
    'Best LogLoss': round(best['logloss'], 4),
    'Avg sec/epoch': round(np.mean([h['time_s'] for h in history]), 1),
}])
print(summary.to_string(index=False))
out_path = '/kaggle/working/results-bloom-dcn.csv'
try:
    summary.to_csv(out_path, index=False); print('Saved:', out_path)
except Exception as e:
    print('Save skipped:', e)